In [2]:
!pip install datasets pandas scikit-learn matplotlib seaborn torch

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached torch-2.10.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pyarrow-23.0.1-cp314-cp314-win_amd64.whl.metadata (3.1 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.6.0-cp314-cp314-win_amd64.whl.metadata (13 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached pyyaml-6.0.3-cp31

# Parte 1 - Tratamento dos Datasets

## Carregar os Datasets

In [3]:
from datasets import load_dataset

hc3 = load_dataset(
    'Hello-SimpleAI/HC3', 
    revision='refs/convert/parquet'
)

openturingbench = load_dataset(
    'MLNTeam-Unical/OpenTuringBench', 
    revision='refs/convert/parquet'
)

claude3_opus = load_dataset(
    'nothingiisreal/Claude-3-Opus-Instruct-15K', 
    'Instruct Data v2 - Merged'
)

print(hc3['train'][0])
print(openturingbench['train'][0])
print(claude3_opus['train'][0])

c:\Users\Carlos\Documents\ProjetoAP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'id': '0', 'question': 'Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n\'t there only be one " # 1 " best seller ? Please explain like I\'m five.', 'human_answers': ['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .', "If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good 

## Processar e uniformizar datasets

In [4]:
import pandas as pd

# Lista vazia que vai guardar todas as linhas uniformizadas
dados_finais = []

# ---------------------------------------------------------
# 1. Processar HC3
# ---------------------------------------------------------
for linha in hc3['train']:
    linha = dict(linha) 
    # Textos humanos
    for resposta_humana in linha['human_answers']:
        dados_finais.append({
            'Text': resposta_humana, 
            'Label': 'Human'
        })
    
    # Textos da IA (ChatGPT)
    for resposta_ai in linha['chatgpt_answers']:
        dados_finais.append({
            'Text': resposta_ai, 
            'Label': 'ChatGPT'
        })

# ---------------------------------------------------------
# 2. Processar OpenTuringBench
# ---------------------------------------------------------
for linha in openturingbench['train']:
    linha = dict(linha) 

    modelo = linha['model']
    if 'llama' in modelo.lower():
        modelo = 'Meta'
        
    dados_finais.append({
        'Text': linha['content'], 
        'Label': modelo
    })

# ---------------------------------------------------------
# 3. Processar Claude-3-Opus
# ---------------------------------------------------------
for linha in claude3_opus['train']:
    linha = dict(linha) 

    dados_finais.append({
        'Text': linha['response'],
        'Label': 'Anthropic'
    })

# ---------------------------------------------------------
# JUNTAR, FORMATAR E GUARDAR
# ---------------------------------------------------------
# Converter para DataFrame
df = pd.DataFrame(dados_finais)

# Shuffle dos dados para misturar as diferentes origens
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

labels_unicos = df['Label'].unique()
print("Labels :", labels_unicos)

Labels : <ArrowStringArray>
[         'Intel/neural-chat-7b-v3-3',  'upstage/SOLAR-10.7B-Instruct-v1.0',
    'microsoft/Phi-3.5-mini-instruct',                            'ChatGPT',
                               'Meta', 'mistralai/Mistral-7B-Instruct-v0.3',
               'google/gemma-2-9b-it',                              'Human',
           'Qwen/Qwen2.5-7B-Instruct',                          'Anthropic']
Length: 10, dtype: str


## Uniformizar Labels

In [5]:
# 1. Criar o dicionário de mapeamento (o que queremos manter e como se vai chamar)
mapeamento_modelos = {
    'Meta': 'Meta',
    'google/gemma-2-9b-it': 'Google',
    'Gemini': 'Google',
    'Anthropic': 'Anthropic',
    'Claude': 'Anthropic',
    'ChatGPT': 'OpenAI',
    'GPT-4': 'OpenAI',
    'Human': 'Human'
}

# 2. Aplicar o mapeamento à coluna 'Label'
# O que não estiver no dicionário (Qwen, Intel, Mistral, etc.) vai ficar 'NaN'
df['Label'] = df['Label'].map(mapeamento_modelos)

# 3. Remover todas as linhas que ficaram com 'NaN' (ou seja, apagar os modelos indesejados)
df = df.dropna(subset=['Label'])

# 4. Confirmar que a limpeza funcionou
print("Labels finais após a limpeza:")
print(df['Label'].unique())

print("\nQuantidade de textos por Label:")
print(df['Label'].value_counts())

Labels finais após a limpeza:
<ArrowStringArray>
['OpenAI', 'Meta', 'Google', 'Human', 'Anthropic']
Length: 5, dtype: str

Quantidade de textos por Label:
Label
Human        117092
OpenAI        53806
Meta          33140
Google        33140
Anthropic      9452
Name: count, dtype: int64


## Garantir limite de palavras

In [6]:
# 1. Função para manter apenas as primeiras 120 palavras
def limitar_palavras(texto, max_palavras=120):
    palavras = str(texto).split()
    return " ".join(palavras[:max_palavras])

# 2. Criar a contagem de palavras e remover APENAS os textos demasiado curtos (< 80)
df['Word_Count'] = df['Text'].apply(lambda x: len(str(x).split()))
df = df[df['Word_Count'] >= 80].copy()

# 3. Cortar os textos que são grandes demais para ficarem com exatamente 120 palavras no máximo
df['Text'] = df['Text'].apply(limitar_palavras)

# 4. Remover a coluna auxiliar
df = df.drop(columns=['Word_Count'])

# Verificar como ficaram as labels agora
print(df['Label'].value_counts())

Label
Human        60580
OpenAI       50838
Google       33140
Meta         33130
Anthropic     9384
Name: count, dtype: int64


In [7]:
import pandas as pd

df.to_csv('df_exportado.csv', index=False)